In [1]:
from pathlib import Path
from os import system
from dataclasses import dataclass
import numpy as np

In [3]:
def get_data_from_mtx(path):
    with path.open('r') as f:
        # Skip comments
        for line in f:
            if not line.startswith('%'):
                break
        
        # Parse the size line which contains: rows cols nonzeros
        try:
            m, n, nnz = map(int, line.split())
            return nnz, m, n
        except ValueError:
            raise ValueError("Invalid Matrix Market file format")

def write_vector(vec : np.ndarray[np.float64], path_out : Path):
    res = "\n".join(vec.astype(str))+"\n"
    path_out.write_text(res)

@dataclass
class MatrixEntry:
  path : Path
  m : int
  n : int
  nnz : int
  
  @classmethod
  def from_path(cls, path: Path):
    nnz, m, n = get_data_from_mtx(path)
    return cls(path=path, m=m, n=n, nnz=nnz)

  def get_data_dir(self):
    return Path("data").joinpath(self.path.stem).joinpath("in")

  def get_out_dir(self):
    return Path("data").joinpath(self.path.stem).joinpath("out")

  def write_vec(self, vec:np.ndarray[np.float64], path_out : Path):
    write_vector(vec=vec, path_out=self.get_data_dir() / path_out)

  def write_x_ones(self,path_out : Path):
    self.write_vec(np.ones(self.n,dtype=np.float64), path_out)
    

In [3]:
arr = ['pre2', 'twotone', 'hcircuit', 'scircuit', 'xenon2', 'lung2', 'stomach', 'torso1', 'torso2', 'torso3','cage12', 'cage13', 
       'matrix_9', 'matrix-new_3', 'barrier2-10', 'barrier2-11', 'barrier2-12', 'barrier2-1', 'barrier2-2', 'barrier2-3','barrier2-4', 
       'barrier2-9', 'ohne2', 'para-10', 'para-4', 'para-5', 'para-6', 'para-7', 'para-8', 'para-9', 'language', 'Hamrle3', 'dc1', 
       'dc2', 'dc3', 'trans4', 'trans5', 'rajat21', 'rajat23', 'rajat24', 'rajat29', 'rajat30',  'ASIC_320k', 'ASIC_320ks','ASIC_680k', 
       'ASIC_680ks',  'FEM_3D_thermal2', 'Baumann', 'crashbasis', 'majorbasis','Raj1', 'hvdc2', 'largebasis', 'tmt_unsym','thermomech_dK', 
       'atmosmodd', 'atmosmodj','atmosmodl','transient', 'PR02R', 'RM07R', 'mac_econ_fwd500', 'webbase-1M','CoupCons3D', 'ML_Laplace',
       'power197k', 'Goodwin_095', 'Goodwin_127','imagesensor', 'power9', 'radiation', 'test1','ss1', 'nxp1', 'marine1','cage14','Transport',
       'nv2', 'dgreen', 'ss', 'vas_stokes_1M']

list_matrices : list[MatrixEntry] = [
  MatrixEntry.from_path(Path(f"/matrices/{matrix}.mtx"))
  for matrix in arr
]
list_matrices.sort(key=lambda x:x.nnz, reverse=True)

path_partitions : list[Path] = [
  Path(f"parts-gpu/{matrix}_2k_s1_gpu_i1")
  for matrix in arr
]

flag = all((
  partition.is_file()
  for partition in path_partitions
))
print(f"All partition vectors {'' if flag else 'do not '}exist")

All partition vectors do not exist


In [4]:
list_matrices[0].write_x_ones("X.txt")

In [5]:
list_matrices[0].get_data_dir()

PosixPath('data/nv2/in')

In [6]:
mat_sample = list_matrices[0]
Y_sample = np.cumsum(np.ones(mat_sample.m,dtype=np.float64))
mat_sample.write_vec(Y_sample, "B.txt")

In [7]:
def run_seq(mat : MatrixEntry):
  path_in = mat.get_data_dir()
  path_out = mat.get_out_dir()
  status = system(
    f"build/mxv-gpu-bicgstab-cpu --n-iters=10 --input-matrix={str(mat.path)} --input-x={str(path_in / 'X.txt')} --input-y={str(path_in / 'B.txt')} --output-x={str(path_out / "X.txt")}"
  )
  return status

In [8]:
run_seq(list_matrices[0])

DEBUG: Sum of all values in matrix is 12594566251836416.000000



	****	Matrix File readed	****Matrix name : /matrices/nv2.mtx
n_iters : 10 
spmv : 6.162527 
file_read : 13.409317 
everything_total : 20.045536

----------------------------------------------------------------------


0

# Create Data of matrix

In [4]:
sample_name = "cage13"

sample_path = Path("/matrices") / f"{sample_name}.mtx"
mat_sample = MatrixEntry.from_path(sample_path)

Y_sample = np.cumsum(np.ones(mat_sample.m,dtype=np.float64))
mat_sample.write_vec(Y_sample, "B.txt")

mat_sample.write_x_ones("X.txt")